# Aula 25 — Árvores de Comportamento (*Behavior Trees*) no SCADA Aqua Index

Motor reativo multi-tick de Behavior Trees em Python 3.13 puro + NumPy para a **Máquina Pneumática de Envasamento de Água em Copos (Aqua Index)**.

Cenários validados com `assert`s:
1. **Cenário 1 — Ciclo Nominal:** `SE-001/p0 == 1`, `Parallel (5/5)` em `E1..E5` com selagem de `2.0 s` (`RUNNING`), guarda anti-colisão `R-01` (`ZSC-401 == 1`) e `Indexar_Mesa_72_Graus` (`M-001`, `RUNNING` -> `SUCCESS`).
2. **Cenário 2 — Pré-aquecimento `RUNNING` (`R-02`):** `TIT-401 < 180 °C` mantém `Aquecer_HT401` em `RUNNING` (`HT-401=1`, `XV-401=0`) até atingir `180 °C`, liberando o ciclo produtivo.
3. **Cenário 3 — Recuperação `Retry(max=2)` de `PIT-301` (`R-03`):** Reacionamento de `VAC-301` com recuperação na retentativa e verificação de esgotamento quando a falha persiste.
4. **Cenário 4 — Preempção Imediata por `HS-004`:** Interrupção (`halt()`) de um ciclo `RUNNING` no tick seguinte ao acionamento de emergência (`M-001=0, M-002=0, HT-401=0, IL-002=1`).

In [1]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, List, Optional, Dict, Any
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

print(f'NumPy {np.__version__} | Matplotlib disponível: {HAS_MPL}')

NumPy 2.5.4 | Matplotlib disponível: False


In [2]:
class Status(Enum):
    SUCCESS = 'SUCCESS'
    FAILURE = 'FAILURE'
    RUNNING = 'RUNNING'


class BTNode:
    def __init__(self, name: str):
        self.name = name
        self.last_status: Optional[Status] = None

    def tick(self, bb: Dict[str, Any]) -> Status:
        raise NotImplementedError

    def halt(self, bb: Dict[str, Any]) -> None:
        self.last_status = None


class Condition(BTNode):
    def __init__(self, name: str, predicate: Callable[[Dict[str, Any]], bool]):
        super().__init__(name)
        self.predicate = predicate

    def tick(self, bb: Dict[str, Any]) -> Status:
        self.last_status = Status.SUCCESS if self.predicate(bb) else Status.FAILURE
        return self.last_status


class Action(BTNode):
    def __init__(
        self,
        name: str,
        fn: Callable[[Dict[str, Any]], Status],
        on_halt: Optional[Callable[[Dict[str, Any]], None]] = None,
    ):
        super().__init__(name)
        self.fn = fn
        self.on_halt = on_halt

    def tick(self, bb: Dict[str, Any]) -> Status:
        self.last_status = self.fn(bb)
        return self.last_status

    def halt(self, bb: Dict[str, Any]) -> None:
        if self.last_status == Status.RUNNING and self.on_halt is not None:
            self.on_halt(bb)
        super().halt(bb)


class Fallback(BTNode):
    def __init__(self, name: str, children: List[BTNode]):
        super().__init__(name)
        self.children = children

    def tick(self, bb: Dict[str, Any]) -> Status:
        for idx, child in enumerate(self.children):
            status = child.tick(bb)
            if status != Status.FAILURE:
                # Preempta qualquer ramo de menor prioridade que estivesse RUNNING
                for other in self.children[idx + 1:]:
                    other.halt(bb)
                self.last_status = status
                return status
        self.last_status = Status.FAILURE
        return Status.FAILURE

    def halt(self, bb: Dict[str, Any]) -> None:
        for child in self.children:
            child.halt(bb)
        super().halt(bb)


class Sequence(BTNode):
    def __init__(self, name: str, children: List[BTNode]):
        super().__init__(name)
        self.children = children

    def tick(self, bb: Dict[str, Any]) -> Status:
        for idx, child in enumerate(self.children):
            status = child.tick(bb)
            if status != Status.SUCCESS:
                for other in self.children[idx + 1:]:
                    other.halt(bb)
                self.last_status = status
                return status
        self.last_status = Status.SUCCESS
        return Status.SUCCESS

    def halt(self, bb: Dict[str, Any]) -> None:
        for child in self.children:
            child.halt(bb)
        super().halt(bb)


class Parallel(BTNode):
    def __init__(self, name: str, children: List[BTNode], success_threshold: Optional[int] = None):
        super().__init__(name)
        self.children = children
        self.m = len(children) if success_threshold is None else success_threshold

    def tick(self, bb: Dict[str, Any]) -> Status:
        statuses = [child.tick(bb) for child in self.children]
        n_succ = sum(1 for s in statuses if s == Status.SUCCESS)
        n_fail = sum(1 for s in statuses if s == Status.FAILURE)
        if n_succ >= self.m:
            self.last_status = Status.SUCCESS
        elif n_fail > len(self.children) - self.m:
            self.last_status = Status.FAILURE
        else:
            self.last_status = Status.RUNNING
        return self.last_status

    def halt(self, bb: Dict[str, Any]) -> None:
        for child in self.children:
            child.halt(bb)
        super().halt(bb)


class Retry(BTNode):
    def __init__(self, name: str, child: BTNode, max_retries: int = 2):
        super().__init__(name)
        self.child = child
        self.max_retries = max_retries
        self.attempts = 0

    def tick(self, bb: Dict[str, Any]) -> Status:
        while self.attempts <= self.max_retries:
            status = self.child.tick(bb)
            if status == Status.SUCCESS:
                self.attempts = 0
                self.last_status = Status.SUCCESS
                return Status.SUCCESS
            if status == Status.RUNNING:
                self.last_status = Status.RUNNING
                return Status.RUNNING
            # FAILURE: contabiliza retentativa
            self.attempts += 1
            if self.attempts <= self.max_retries:
                bb['vac_retries_used'] = self.attempts
                self.last_status = Status.RUNNING
                return Status.RUNNING
        self.attempts = 0
        self.last_status = Status.FAILURE
        return Status.FAILURE

    def halt(self, bb: Dict[str, Any]) -> None:
        self.attempts = 0
        self.child.halt(bb)
        super().halt(bb)


class Inverter(BTNode):
    def __init__(self, name: str, child: BTNode):
        super().__init__(name)
        self.child = child

    def tick(self, bb: Dict[str, Any]) -> Status:
        s = self.child.tick(bb)
        if s == Status.SUCCESS:
            self.last_status = Status.FAILURE
        elif s == Status.FAILURE:
            self.last_status = Status.SUCCESS
        else:
            self.last_status = Status.RUNNING
        return self.last_status

    def halt(self, bb: Dict[str, Any]) -> None:
        self.child.halt(bb)
        super().halt(bb)


class Timeout(BTNode):
    def __init__(self, name: str, child: BTNode, max_ticks: int):
        super().__init__(name)
        self.child = child
        self.max_ticks = max_ticks
        self.elapsed = 0

    def tick(self, bb: Dict[str, Any]) -> Status:
        s = self.child.tick(bb)
        if s == Status.RUNNING:
            self.elapsed += 1
            if self.elapsed > self.max_ticks:
                self.child.halt(bb)
                self.elapsed = 0
                self.last_status = Status.FAILURE
                return Status.FAILURE
        else:
            self.elapsed = 0
        self.last_status = s
        return s

    def halt(self, bb: Dict[str, Any]) -> None:
        self.elapsed = 0
        self.child.halt(bb)
        super().halt(bb)

In [3]:
def criar_blackboard_nominal() -> Dict[str, Any]:
    return {
        'dt_tick': 0.5,          # 0.5 s por tick
        'tick_idx': 0,
        # Entradas ISA-5.1
        'HS-004': 0,             # Botão de emergência
        'R-06': 0,               # Contradição elétrica em fim de curso (ZSC-x ∧ ZSO-x)
        'ZS-102': 1,             # Copo presente em E1
        'REQ_tampa': 0,          # Solicitação de captura de tampa em E3
        'PIT-301': 1,            # Pressostato de vácuo (tampa capturada)
        'TIT-401': 185.0,        # Temperatura cabeçote E4 (°C)
        'SE-001/p0': 1,          # Mesa posicionada no ângulo nominal
        'ZSC-401': 1,            # Prensa F recuada (guarda R-01)
        'ZSO-401': 0,            # Prensa F avançada
        # Saídas / Atuadores ISA-5.1
        'M-001': 0,              # Motor mesa indexadora
        'M-002': 1,              # Motor esteira de saída
        'HT-401': 1,             # Resistência cartucho
        'IL-002': 0,             # LED emergência
        'XV-401': 0,             # Prensa cil. F
        'VAC-301': 0,            # Válvula de vácuo
        # Estado interno dos timers e eventos
        'selagem_tempo_s': 0.0,
        'selagem_concluida': False,
        'index_angulo_deg': 0.0,
        'e1_done': False,
        'e2_done': False,
        'e3_done': False,
        'e5_done': False,
        'vac_retries_used': 0,
        'vac_falhas_restantes': 0,
        'modo_atual': 'INIT',
        'alarme_ihm': None,
        'halted_nodes': [],
    }


def acao_parada_segura(bb: Dict[str, Any]) -> Status:
    bb['M-001'] = 0
    bb['M-002'] = 0
    bb['HT-401'] = 0
    bb['XV-401'] = 0
    bb['VAC-301'] = 0
    bb['IL-002'] = 1
    bb['modo_atual'] = 'PARADA_SEGURA'
    bb['alarme_ihm'] = 'EMERGENCIA_HS004_OU_R06'
    return Status.SUCCESS


def acao_pausar_mag100(bb: Dict[str, Any]) -> Status:
    bb['M-001'] = 0
    bb['modo_atual'] = 'PAUSA_MAG100'
    bb['alarme_ihm'] = 'SOLICITAR_REABASTECIMENTO_MAG100'
    return Status.RUNNING


def acao_vac301_verificar(bb: Dict[str, Any]) -> Status:
    bb['VAC-301'] = 1
    bb['modo_atual'] = 'RECUPERACAO_VACUO_E3'
    if bb['vac_falhas_restantes'] > 0:
        bb['vac_falhas_restantes'] -= 1
        bb['PIT-301'] = 0
        return Status.FAILURE
    bb['PIT-301'] = 1
    bb['REQ_tampa'] = 0
    bb['alarme_ihm'] = None
    return Status.SUCCESS


def acao_aquecer_ht401(bb: Dict[str, Any]) -> Status:
    bb['HT-401'] = 1
    bb['XV-401'] = 0
    bb['M-001'] = 0
    bb['modo_atual'] = 'AQUECIMENTO_HT401'
    bb['TIT-401'] = min(185.0, bb['TIT-401'] + 5.0)
    return Status.RUNNING


def acao_e1(bb: Dict[str, Any]) -> Status:
    bb['e1_done'] = True
    return Status.SUCCESS


def acao_e2(bb: Dict[str, Any]) -> Status:
    bb['e2_done'] = True
    return Status.SUCCESS


def acao_e3(bb: Dict[str, Any]) -> Status:
    bb['e3_done'] = True
    return Status.SUCCESS


def acao_e4_selagem_2s(bb: Dict[str, Any]) -> Status:
    if bb['selagem_concluida']:
        bb['XV-401'] = 0
        bb['ZSC-401'] = 1
        bb['ZSO-401'] = 0
        return Status.SUCCESS
    bb['modo_atual'] = 'CICLO_E1_E5_SELAGEM'
    bb['XV-401'] = 1
    bb['ZSC-401'] = 0
    bb['ZSO-401'] = 1
    bb['selagem_tempo_s'] += bb['dt_tick']
    if bb['selagem_tempo_s'] >= 2.0:
        bb['selagem_concluida'] = True
        bb['XV-401'] = 0
        bb['ZSC-401'] = 1
        bb['ZSO-401'] = 0
        return Status.SUCCESS
    return Status.RUNNING


def halt_e4(bb: Dict[str, Any]) -> None:
    bb['XV-401'] = 0
    bb['halted_nodes'].append('E4_Termosselar_2s')


def acao_e5(bb: Dict[str, Any]) -> Status:
    bb['M-002'] = 1
    bb['e5_done'] = True
    return Status.SUCCESS


def acao_indexar_mesa(bb: Dict[str, Any]) -> Status:
    bb['modo_atual'] = 'INDEXANDO_MESA_72_GRAUS'
    bb['M-001'] = 1
    bb['SE-001/p0'] = 0
    bb['index_angulo_deg'] += 36.0
    if bb['index_angulo_deg'] >= 72.0:
        bb['M-001'] = 0
        bb['SE-001/p0'] = 1
        bb['index_angulo_deg'] = 0.0
        bb['selagem_tempo_s'] = 0.0
        bb['selagem_concluida'] = False
        bb['modo_atual'] = 'CICLO_CONCLUIDO'
        return Status.SUCCESS
    return Status.RUNNING


def halt_indexar(bb: Dict[str, Any]) -> None:
    bb['M-001'] = 0
    bb['halted_nodes'].append('Indexar_Mesa_72_Graus')


def construir_bt_aqua_index() -> Fallback:
    ramo_seguranca = Sequence('Seguranca_Global', [
        Condition('Emergencia_ou_R06', lambda bb: bb['HS-004'] == 1 or bb['R-06'] == 1),
        Action('Parada_Segura', acao_parada_segura),
    ])

    ramo_recuperacao = Fallback('Recuperacao_Operacional', [
        Sequence('R04_Falta_Copo', [
            Condition('Not_ZS102', lambda bb: bb['ZS-102'] == 0),
            Action('Pausar_e_Solicitar_MAG100', acao_pausar_mag100),
        ]),
        Sequence('R03_Vacuo_E3', [
            Condition('Req_Tampa_Sem_PIT301', lambda bb: bb['REQ_tampa'] == 1 and bb['PIT-301'] == 0),
            Retry('Retry_VAC301_max2', Action('Acionar_VAC301', acao_vac301_verificar), max_retries=2),
        ]),
        Sequence('R02_Temperatura_E4', [
            Condition('TIT401_Abaixo_180C', lambda bb: bb['TIT-401'] < 180.0),
            Action('Aquecer_HT401', acao_aquecer_ht401),
        ]),
    ])

    ramo_produtivo = Sequence('Ciclo_Produtivo', [
        Condition('Mesa_Posicionada_SE001', lambda bb: bb['SE-001/p0'] == 1 or bb['index_angulo_deg'] > 0.0),
        Parallel('E1_a_E5_Paralelo', [
            Action('E1_Dispensar_Copo', acao_e1),
            Action('E2_Envasar_150mL', acao_e2),
            Action('E3_Posicionar_Tampa', acao_e3),
            Action('E4_Termosselar_2s', acao_e4_selagem_2s, on_halt=halt_e4),
            Action('E5_Ejetar_Copo', acao_e5),
        ], success_threshold=5),
        Condition('Guarda_R01_ZSC401', lambda bb: bb['ZSC-401'] == 1),
        Action('Indexar_Mesa_72_Graus', acao_indexar_mesa, on_halt=halt_indexar),
    ])

    return Fallback('Raiz_Aqua_Index', [ramo_seguranca, ramo_recuperacao, ramo_produtivo])

In [4]:
# ============================================================================
# CENÁRIO 1 — Ciclo Nominal (Parallel E1..E5 com selagem 2s + R-01 + Index 72°)
# ============================================================================
bt1 = construir_bt_aqua_index()
bb1 = criar_blackboard_nominal()
hist1 = []
for t in range(1, 6):
    bb1['tick_idx'] = t
    st = bt1.tick(bb1)
    hist1.append((t, st.value, bb1['modo_atual'], bb1['selagem_tempo_s'], bb1['M-001']))
    print(f'[C1 tick={t}] status={st.value:<7} modo={bb1["modo_atual"]:<24} selagem={bb1["selagem_tempo_s"]:.1f}s M-001={bb1["M-001"]}')

assert hist1[0][1] == 'RUNNING' and hist1[0][2] == 'CICLO_E1_E5_SELAGEM'
assert hist1[3][1] == 'RUNNING' and hist1[3][2] == 'INDEXANDO_MESA_72_GRAUS'
assert hist1[4][1] == 'SUCCESS' and hist1[4][2] == 'CICLO_CONCLUIDO'
assert bb1['e1_done'] and bb1['e2_done'] and bb1['e3_done'] and bb1['e5_done']
assert bb1['ZSC-401'] == 1 and bb1['M-001'] == 0
print('Cenário 1 OK!\n')

# ============================================================================
# CENÁRIO 2 — Pré-aquecimento RUNNING (TIT-401 < 180 °C -> R-02)
# ============================================================================
bt2 = construir_bt_aqua_index()
bb2 = criar_blackboard_nominal()
bb2['TIT-401'] = 165.0  # Abaixo de 180 °C
hist2 = []
for t in range(1, 6):
    bb2['tick_idx'] = t
    st = bt2.tick(bb2)
    hist2.append((t, st.value, bb2['modo_atual'], bb2['TIT-401']))
    print(f'[C2 tick={t}] status={st.value:<7} modo={bb2["modo_atual"]:<24} TIT-401={bb2["TIT-401"]:.1f}°C')

assert all(h[1] == 'RUNNING' and h[2] == 'AQUECIMENTO_HT401' for h in hist2[:3])
assert hist2[3][2] == 'CICLO_E1_E5_SELAGEM' and bb2['TIT-401'] >= 180.0
print('Cenário 2 OK!\n')

# ============================================================================
# CENÁRIO 3 — Recuperação Retry(max=2) de PIT-301 (R-03)
# ============================================================================
bt3 = construir_bt_aqua_index()
bb3 = criar_blackboard_nominal()
bb3['REQ_tampa'] = 1
bb3['PIT-301'] = 0
bb3['vac_falhas_restantes'] = 2  # Falha na tentativa inicial + 1a retentativa; recupera na 2a

st3_1 = bt3.tick(bb3)
r3_1 = bb3['vac_retries_used']
st3_2 = bt3.tick(bb3)
r3_2 = bb3['vac_retries_used']
st3_3 = bt3.tick(bb3)
print(f'[C3] tick1={st3_1.value} (retries={r3_1}), tick2={st3_2.value} (retries={r3_2}), tick3={st3_3.value}, PIT-301={bb3["PIT-301"]}')

assert st3_1 == Status.RUNNING and r3_1 == 1
assert st3_2 == Status.RUNNING and r3_2 == 2
assert st3_3 == Status.SUCCESS and bb3['PIT-301'] == 1 and bb3['REQ_tampa'] == 0

# Teste de esgotamento de Retry(max=2) quando falha persiste
bt3_fail = construir_bt_aqua_index()
bb3_fail = criar_blackboard_nominal()
bb3_fail['REQ_tampa'] = 1
bb3_fail['PIT-301'] = 0
bb3_fail['vac_falhas_restantes'] = 5
bt3_fail.tick(bb3_fail)  # tentativa 0 -> falha, retry 1
bt3_fail.tick(bb3_fail)  # tentativa 1 -> falha, retry 2
st_esgotado = bt3_fail.children[1].children[1].tick(bb3_fail)  # ramo R03 retorna FAILURE
assert st_esgotado == Status.FAILURE
print('Cenário 3 OK!\n')

# ============================================================================
# CENÁRIO 4 — Preempção Imediata por HS-004 durante RUNNING
# ============================================================================
bt4 = construir_bt_aqua_index()
bb4 = criar_blackboard_nominal()
st4_1 = bt4.tick(bb4)  # Entra em E4_Termosselar_2s (RUNNING)
assert st4_1 == Status.RUNNING and bb4['XV-401'] == 1

bb4['HS-004'] = 1  # Operador aciona botão de emergência no tick 2
st4_2 = bt4.tick(bb4)
print(f'[C4] tick1={st4_1.value} (XV-401=1) -> tick2 HS-004=1: status={st4_2.value}, modo={bb4["modo_atual"]}, halted={bb4["halted_nodes"]}')

assert st4_2 == Status.SUCCESS
assert bb4['modo_atual'] == 'PARADA_SEGURA'
assert bb4['M-001'] == 0 and bb4['M-002'] == 0 and bb4['HT-401'] == 0 and bb4['IL-002'] == 1
assert 'E4_Termosselar_2s' in bb4['halted_nodes']
print('Cenário 4 OK!')

[C1 tick=1] status=RUNNING modo=CICLO_E1_E5_SELAGEM      selagem=0.5s M-001=0
[C1 tick=2] status=RUNNING modo=CICLO_E1_E5_SELAGEM      selagem=1.0s M-001=0
[C1 tick=3] status=RUNNING modo=CICLO_E1_E5_SELAGEM      selagem=1.5s M-001=0
[C1 tick=4] status=RUNNING modo=INDEXANDO_MESA_72_GRAUS  selagem=2.0s M-001=1
[C1 tick=5] status=SUCCESS modo=CICLO_CONCLUIDO          selagem=0.0s M-001=0
Cenário 1 OK!

[C2 tick=1] status=RUNNING modo=AQUECIMENTO_HT401        TIT-401=170.0°C
[C2 tick=2] status=RUNNING modo=AQUECIMENTO_HT401        TIT-401=175.0°C
[C2 tick=3] status=RUNNING modo=AQUECIMENTO_HT401        TIT-401=180.0°C
[C2 tick=4] status=RUNNING modo=CICLO_E1_E5_SELAGEM      TIT-401=180.0°C
[C2 tick=5] status=RUNNING modo=CICLO_E1_E5_SELAGEM      TIT-401=180.0°C
Cenário 2 OK!

[C3] tick1=RUNNING (retries=1), tick2=RUNNING (retries=2), tick3=SUCCESS, PIT-301=1
Cenário 3 OK!

[C4] tick1=RUNNING (XV-401=1) -> tick2 HS-004=1: status=SUCCESS, modo=PARADA_SEGURA, halted=['E4_Termosselar_2s']
Ce